In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import sklearn

In [2]:
# read 2023 data
data_2023 = pd.read_csv('Data/2023_stats/2023_data.csv')
ken_2023 = pd.read_csv('Data/2023_stats/2023_Kenpom.csv')
res_2023 = pd.read_csv('Data/2023_stats/2023_results.csv')

In [3]:
# read 2024 data
data_2024 = pd.read_csv('Data/2024_stats/2024_data.csv')
ken_2024 = pd.read_csv('Data/2024_stats/2024_Kenpom.csv')
res_2024 = pd.read_csv('Data/2024_stats/2024_results.csv')

In [4]:
res_2023.head() # check formatting by lookign at head

,Team 1 Name,Team 1 Score,Team 2 Name,Team 2 Score
0,FDU,84,Texas Southern,61
1,Arizona St.,98,Nevada,73
2,Texas A&M - CC,75,SE Missouri St.,71
3,Pittsburgh,60,Mississippi St.,59
4,Houston,63,North Kentucky,52


In [5]:
import random
def swap(df):
    """
    Randomly swap the team 1 columns and the team 2 columns so team 1 isn't seen as favorite
    :param df: pandas data frame of head to head games with team name and scores
    :return: pandas data frame head to head games with team name and scores
    """
    for i in range(len(df)):
        int = random.choice([0, 1])
        if int == 0:
            df.loc[i, 'Team 1 Score'], df.loc[i, 'Team 2 Score'] = df.loc[i, 'Team 2 Score'], df.loc[i, 'Team 1 Score']
            df.loc[i, 'Team 1 Name'], df.loc[i, 'Team 2 Name'] = df.loc[i, 'Team 2 Name'], df.loc[i, 'Team 1 Name']
    
    return df

In [6]:
swap(res_2023)

,Team 1 Name,Team 1 Score,Team 2 Name,Team 2 Score
0,FDU,84,Texas Southern,61
1,Nevada,73,Arizona St.,98
2,Texas A&M - CC,75,SE Missouri St.,71
3,Mississippi St.,59,Pittsburgh,60
4,North Kentucky,52,Houston,63
...,...,...,...,...
62,Kansas St.,76,FAU,79
63,Creighton,56,San Diego St.,57
64,San Diego St.,72,FAU,71
65,Miami FL,59,UConn,72


In [7]:
swap(res_2024)

,Team 1 Name,Team 1 Score,Team 2 Name,Team 2 Score
0,Howard,68,Wagner,71
1,Colorado St.,67,Virginia,42
2,Montana St.,81,Grambling St.,88
3,Colorado,60,Boise St.,53
4,UConn,91,Stetson,52
...,...,...,...,...
62,Duke,64,NC State,76
63,Purdue,72,Tennessee,66
64,UConn,86,Alabama,72
65,Purdue,63,NC State,50


In [8]:
def make_games(res, ken):
    team_stats = ken.set_index('Team').to_dict(orient='index') # convert to dictionary for lookup
    
    game_data = []
    
    # iterate through each game in results_df
    for _, row in res.iterrows():
        team1 = row['Team 1 Name']
        team2 = row['Team 2 Name']
        team1_score = row['Team 1 Score']
        team2_score = row['Team 2 Score']
    
        if team1 in team_stats and team2 in team_stats:
            # get both teams stats
            team1_stats = team_stats[team1]
            team2_stats = team_stats[team2]
    
            # calculate the difference for each stat for the team
            stat_diffs = {stat: float(team1_stats[stat]) - float(team2_stats[stat]) 
                          for stat in team1_stats.keys() if isinstance(team1_stats[stat], (int, float))}
    
            # append results
            game_data.append({
                'Team 1 Name': team1,
                'Team 1 Score': team1_score,
                'Team 2 Name': team2,
                'Team 2 Score': team2_score,
                **stat_diffs
            })
        else:
            print(team1 + " not found")
    
    games_df = pd.DataFrame(game_data) # convert to a data frame and return
    
    return games_df

In [9]:
def add_score(df):
    df['Score'] = df['Team 1 Score'] - df['Team 2 Score']
    
    return df

In [10]:
games_2023 = make_games(res_2023, ken_2023)
games_2024 = make_games(res_2024, ken_2024)

In [11]:
add_score(games_2023)
add_score(games_2024)

,Team 1 Name,Team 1 Score,Team 2 Name,Team 2 Score,Rk,NetRtg,ORtg,DRtg,AdjT,Luck,Opp NetRtg,Opp ORtg,Opp DRtg,Non Conf Opp NetRtg,Score
0,Howard,68,Wagner,71,-17.0,1.62,9.2,7.5,6.2,-0.032,0.45,-0.2,-0.7,1.09,-3
1,Colorado St.,67,Virginia,42,-32.0,5.22,8.9,3.7,6.0,-0.134,0.23,0.3,0.1,6.93,25
2,Montana St.,81,Grambling St.,88,-36.0,2.44,3.4,1.0,4.2,-0.148,0.84,3.2,2.4,-14.92,-7
3,Colorado,60,Boise St.,53,-20.0,3.06,5.0,1.9,1.0,0.016,-0.21,-0.2,0.1,-9.55,7
4,UConn,91,Stetson,52,-219.0,40.94,17.7,-23.2,-1.9,-0.108,15.28,7.5,-7.8,-7.98,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,Duke,64,NC State,76,-38.0,10.57,7.3,-3.2,-1.6,-0.078,-1.26,-1.1,0.3,4.95,-12
63,Purdue,72,Tennessee,66,-2.0,4.01,8.4,4.4,-2.3,0.074,1.30,-0.2,-1.4,1.61,6
64,UConn,86,Alabama,72,-13.0,13.47,1.5,-11.9,-8.0,0.038,-2.29,-1.9,0.4,-12.86,14
65,Purdue,63,NC State,50,-42.0,14.72,10.9,-3.8,-1.0,0.034,3.32,2.2,-1.0,15.57,13


In [12]:
games_2023.columns

Index(['Team 1 Name', 'Team 1 Score', 'Team 2 Name', 'Team 2 Score', 'Rk',
       'NetRtg', 'ORtg', 'DRtg', 'AdjT', 'Luck', 'Opp NetRtg', 'Opp ORtg',
       'Opp DRtg', 'Non Conf Opp NetRtg', 'Score'],
      dtype='object')

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.metrics import accuracy_score

def run(X_train, y_train, X_test, y_test):
    
    selector = SelectKBest(score_func=f_classif, k=2)
    X_train_selected = selector.fit_transform(X_train, y_train)
    X_test_selected = selector.transform(X_test)
    model = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=5)
    model.fit(X_train_selected, y_train)
    
    y_pred = model.predict(X_test_selected)
    
    accuracy = accuracy_score(y_test, y_pred)
    selected_features = X_train.columns[selector.get_support()]
    
    return accuracy, selected_features
    

In [14]:
X1 = games_2023.drop(columns=['Team 1 Name', 'Team 1 Score', 'Team 2 Name', 'Team 2 Score', 'Score'])
y1 = (games_2023['Score'] > 0).astype(int)

X2 = games_2024.drop(columns=['Team 1 Name', 'Team 1 Score', 'Team 2 Name', 'Team 2 Score', 'Score'])
y2 = (games_2024['Score'] > 0).astype(int)

In [15]:
run(X1, y1, X2, y2)

(0.7014925373134329, Index(['NetRtg', 'ORtg'], dtype='object'))

In [16]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC


def ensemble_prediction(X_train, y_train, X_test, y_test):
    from sklearn.ensemble import VotingClassifier
    from sklearn.linear_model import LogisticRegression
    
    # Create base models
    xgb_model = xgb.XGBClassifier(random_state=42, n_estimators=500)
    lr_model  = LogisticRegression(random_state=42)
    rf_model  = RandomForestClassifier(n_estimators=500, random_state=42)
    svm_model = SVC(probability=True, random_state=42)
    gbm_model = GradientBoostingClassifier(n_estimators=500, random_state=42)
    knn_model = KNeighborsClassifier(n_neighbors=5)
    
    # Create ensemble
    ensemble = VotingClassifier(
        estimators=[
            ('xgb', xgb_model),
            ('lr', lr_model),
            ('rf', rf_model),
            ('svm', svm_model),
            #('gbm', gbm_model),
            ('knn', knn_model),
            
        ],
        voting='soft'
    )
    
    # Train and evaluate
    fit = ensemble.fit(X_train, y_train)
    y_pred = ensemble.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    return accuracy, fit

In [17]:
acc, fit = ensemble_prediction(X1, y1, X2, y2)
print(acc)

0.746268656716418


In [18]:
fit.predict(X2.iloc[[0]])[0]

0

In [19]:
correct = 0
for i in range(X2.shape[0]):
    pred = fit.predict(X2.iloc[[i]])[0]
    real = y2.iloc[i]
    if pred == real:
        correct += 1
        
print(correct/X2.shape[0])

0.746268656716418


In [20]:
def simulate_tournament(initial_matchups, kendf, model):
    """
    Simulates a March Madness tournament based on initial matchups.
    
    Parameters:
    initial_matchups (DataFrame): DataFrame with columns 'Team 1 Name' and 'Team 2 Name'
                                 representing the first round matchups
    kendf (DataFrame): KenPom data for each team
    model: Trained prediction model
    
    Returns:
    dict: Tournament results with winners for each round
    """
    import pandas as pd
    import copy
    
    # Validate input
    if 'Team 1 Name' not in initial_matchups.columns or 'Team 2 Name' not in initial_matchups.columns:
        raise ValueError("Initial matchups DataFrame must have 'Team 1 Name' and 'Team 2 Name' columns")
    
    num_teams = len(initial_matchups) * 2
    
    # Determine tournament structure based on number of teams
    if num_teams == 64:
        rounds = ['Round 1', 'Round 2', 'Sweet 16', 'Elite 8', 'Final 4', 'Championship']
    elif num_teams == 32:
        rounds = ['Round 1', 'Sweet 16', 'Elite 8', 'Final 4', 'Championship']
    elif num_teams == 16:
        rounds = ['Round 1', 'Elite 8', 'Final 4', 'Championship']
    elif num_teams == 8:
        rounds = ['Round 1', 'Final 4', 'Championship']
    elif num_teams == 4:
        rounds = ['Round 1', 'Championship']
    else:
        raise ValueError(f"Unsupported number of teams: {num_teams}")
    
    # Dictionary to store results of each round
    results = {round_name: None for round_name in rounds}
    results['Champion'] = None
    
    # Start with initial matchups
    current_bracket = initial_matchups.copy()
    results[rounds[0]] = current_bracket.copy()
    
    # Simulate each round
    for i, round_name in enumerate(rounds):
        print(f"Simulating {round_name}...")
        next_round_teams = []
        
        # For each matchup in current round
        for idx, row in current_bracket.iterrows():
            # Get the teams
            team1 = row['Team 1 Name']
            team2 = row['Team 2 Name']
            
            # Create a single game DataFrame for prediction
            game = pd.DataFrame({
                'Team 1 Name': [team1],
                'Team 2 Name': [team2],
                'Team 1 Score': [0],  # Placeholder
                'Team 2 Score': [0]   # Placeholder
            })
            
            # Process the game using make_games and prepare for prediction
            processed_game = make_games(game, kendf)
            
            # Check if teams were found in KenPom data
            if processed_game.empty:
                print(f"Warning: Could not find KenPom data for matchup: {team1} vs {team2}")
                # Choose a random winner if data is missing
                import random
                winner = team1 if random.random() > 0.5 else team2
            else:
                processed_game = add_score(processed_game)
                X_game = processed_game.drop(columns=['Team 1 Name', 'Team 1 Score', 'Team 2 Name', 'Team 2 Score', 'Score'])
                
                # Predict the winner
                prediction = model.predict(X_game)[0]
                
                # Determine the winner based on prediction
                winner = team1 if prediction == 1 else team2
            
            next_round_teams.append(winner)
        
        # If this is the final round, store the champion and exit
        if i == len(rounds) - 1:
            results['Champion'] = next_round_teams[0]
            break
            
        # Create bracket for the next round
        next_round_bracket = pd.DataFrame({
            'Team 1 Name': next_round_teams[0::2],
            'Team 2 Name': next_round_teams[1::2]
        })
        
        # Update current bracket and store results
        current_bracket = next_round_bracket
        results[rounds[i+1]] = current_bracket.copy()
    
    return results


In [21]:
def print_tournament_results(results):
    """
    Prints the results of the tournament in a readable format.
    
    Parameters:
    results (dict): Tournament results from simulate_tournament
    """
    print("\n🏀 MARCH MADNESS TOURNAMENT SIMULATION RESULTS 🏀\n")
    
    # Print each round's matchups and winners
    for round_name, bracket in results.items():
        if round_name == 'Champion':
            print(f"\n🏆 CHAMPION: {results['Champion']} 🏆")
        elif bracket is not None:
            print(f"\n== {round_name} ==")
            for i, row in bracket.iterrows():
                print(f"Game {i+1}: {row['Team 1 Name']} vs {row['Team 2 Name']}")

In [22]:
def get_bracket_visual(results):
    """
    Creates a simple text-based visualization of the tournament bracket.
    
    Parameters:
    results (dict): Tournament results from simulate_tournament
    
    Returns:
    str: Text representation of the bracket
    """
    output = []
    rounds = [r for r in results.keys() if r != 'Champion']
    
    # Get the champion
    champion = results['Champion']
    
    # Generate bracket visualization
    output.append("TOURNAMENT BRACKET")
    output.append("=================")
    
    for i, round_name in enumerate(rounds):
        output.append(f"\n{round_name}:")
        bracket = results[round_name]
        
        if bracket is None:
            continue
            
        for j, row in bracket.iterrows():
            team1 = row['Team 1 Name']
            team2 = row['Team 2 Name']
            
            # Check if we know the winner of this matchup
            winner = None
            if i < len(rounds) - 1:
                next_round = results[rounds[i+1]]
                if next_round is not None:
                    # Check if either team appears in the next round
                    if any(next_round['Team 1 Name'] == team1) or any(next_round['Team 2 Name'] == team1):
                        winner = team1
                    elif any(next_round['Team 1 Name'] == team2) or any(next_round['Team 2 Name'] == team2):
                        winner = team2
            elif i == len(rounds) - 1:
                # Championship game
                winner = champion
                
            # Format the matchup line
            if winner:
                if winner == team1:
                    output.append(f"  {team1} ✓ vs {team2}")
                else:
                    output.append(f"  {team1} vs {team2} ✓")
            else:
                output.append(f"  {team1} vs {team2}")
    
    output.append(f"\nChampion: 🏆 {champion} 🏆")
    
    return "\n".join(output)

In [23]:
teams = games_2024[['Team 1 Name', 'Team 2 Name']].iloc[4:36]

In [24]:
teams

,Team 1 Name,Team 2 Name
4,UConn,Stetson
5,Florida Atlantic,Northwestern
6,San Diego St.,UAB
7,Yale,Auburn
8,Duquesne,BYU
9,Illinois,Morehead St.
10,Washington St.,Drake
11,Iowa St.,South Dakota St.
12,Wagner,North Carolina
13,Mississippi St.,Michigan St.


In [25]:
def run_simulation(initial_matchups_df, kenpom_df, model):
    """
    Run a tournament simulation with the provided initial matchups.
    
    Parameters:
    initial_matchups_df (DataFrame): DataFrame with Team 1 Name and Team 2 Name columns
    kenpom_df (DataFrame): KenPom data for all teams
    model: Trained prediction model
    
    Returns:
    dict: Tournament results
    """
    # Run the simulation
    results = simulate_tournament(initial_matchups_df, kenpom_df, model)
    
    # Print the results
    print_tournament_results(results)
    
    # Get and print the bracket visualization
    bracket_visual = get_bracket_visual(results)
    print("\n")
    print(bracket_visual)
    
    return results

In [26]:
run_simulation(teams, ken_2024, fit)

Simulating Round 1...
Simulating Round 2...
Simulating Sweet 16...
Simulating Elite 8...
Simulating Final 4...
Simulating Championship...

🏀 MARCH MADNESS TOURNAMENT SIMULATION RESULTS 🏀


== Round 1 ==
Game 5: UConn vs Stetson
Game 6: Florida Atlantic vs Northwestern
Game 7: San Diego St. vs UAB
Game 8: Yale vs Auburn
Game 9: Duquesne vs BYU
Game 10: Illinois vs Morehead St.
Game 11: Washington St. vs Drake
Game 12: Iowa St. vs South Dakota St.
Game 13: Wagner vs North Carolina
Game 14: Mississippi St. vs Michigan St.
Game 15: Saint Mary's vs Grand Canyon
Game 16: Alabama vs Charleston
Game 17: New Mexico vs Clemson
Game 18: Colgate vs Baylor
Game 19: Dayton vs Nevada
Game 20: Arizona vs Long Beach St.
Game 21: Houston vs Longwood
Game 22: Texas A&M vs Nebraska
Game 23: James Madison vs Wisconsin
Game 24: Duke vs Vermont
Game 25: NC State vs Texas Tech
Game 26: Oakland vs Kentucky
Game 27: Florida vs Colorado
Game 28: Western Kentucky vs Marquette
Game 29: Purdue vs Grambling St.
Game

{'Round 1':          Team 1 Name       Team 2 Name
 4              UConn           Stetson
 5   Florida Atlantic      Northwestern
 6      San Diego St.               UAB
 7               Yale            Auburn
 8           Duquesne               BYU
 9           Illinois      Morehead St.
 10    Washington St.             Drake
 11          Iowa St.  South Dakota St.
 12            Wagner    North Carolina
 13   Mississippi St.      Michigan St.
 14      Saint Mary's      Grand Canyon
 15           Alabama        Charleston
 16        New Mexico           Clemson
 17           Colgate            Baylor
 18            Dayton            Nevada
 19           Arizona    Long Beach St.
 20           Houston          Longwood
 21         Texas A&M          Nebraska
 22     James Madison         Wisconsin
 23              Duke           Vermont
 24          NC State        Texas Tech
 25           Oakland          Kentucky
 26           Florida          Colorado
 27  Western Kentucky        

In [27]:
def make_games_scoreless(res, ken):
    # Convert team stats data into a dictionary for quick lookup
    team_stats = ken.set_index('Team').to_dict(orient='index')
    
    # Create a new list to store the transformed data
    game_data = []
    
    # Iterate through each game in results_df
    for _, row in res.iterrows():
        team1 = row['Team 1 Name']
        team2 = row['Team 2 Name']
    
        if team1 in team_stats and team2 in team_stats:
            # Get stats for both teams
            team1_stats = team_stats[team1]
            team2_stats = team_stats[team2]
    
            # Compute the differences in stats (team1 - team2)
            stat_diffs = {stat: float(team1_stats[stat]) - float(team2_stats[stat]) 
                          for stat in team1_stats.keys() if isinstance(team1_stats[stat], (int, float))}
    
            # Append results
            game_data.append({
                'Team 1 Name': team1,
                'Team 2 Name': team2,
                **stat_diffs
            })
        else:
            print(team1 + " not found")
    
    # Convert the list into a DataFrame
    games_df = pd.DataFrame(game_data)
    
    return games_df

In [28]:
matchups_2025 = pd.read_csv('Data/2025_stats/2025_teams.csv')
ken_2025 = pd.read_csv('Data/2025_stats/2025_kenpom.csv')

In [29]:
make_games_scoreless(matchups_2025, ken_2025)

,Team 1 Name,Team 2 Name,Rk,NetRtg,ORtg,DRtg,AdjT,Luck,Opp NetRtg,Opp ORtg,Opp DRtg,Opp Non Conf NetRtg
0,Auburn,Alabama St.,-269.0,44.36,27.1,-17.3,-0.2,-0.031,27.87,16.4,-11.4,4.17
1,Louisville,Creighton,-13.0,3.98,1.3,-2.7,1.7,0.009,-1.09,0.5,1.6,1.70
2,Michigan,UC San Diego,-12.0,3.91,1.0,-2.9,3.8,0.060,16.86,8.8,-8.1,5.88
3,Texas A&M,Yale,-57.0,13.01,1.1,-12.0,-1.0,0.042,18.66,9.0,-9.6,7.86
4,Ole Miss,North Carolina,-4.0,1.47,-1.9,-3.4,-2.4,0.039,3.50,1.9,-1.7,-13.90
5,Iowa St.,Lipscomb,-72.0,17.55,6.8,-10.8,2.7,-0.020,17.24,7.8,-9.4,-4.82
6,Marquette,New Mexico,-14.0,4.66,5.3,0.6,-5.1,-0.013,6.61,4.2,-2.4,3.66
7,Michigan St.,Bryant,-140.0,26.44,11.4,-15.0,-4.7,-0.005,20.44,11.3,-9.1,0.02
8,Florida,Norfolk St.,-178.0,37.69,21.4,-16.2,3.2,-0.049,21.65,11.5,-10.2,-5.07
9,UConn,Oklahoma,-4.0,1.06,2.9,1.9,-4.8,-0.024,-4.46,-2.4,2.0,5.30


In [30]:
new_x = pd.concat([games_2023, games_2024])

In [31]:
X1 = new_x.drop(columns=['Team 1 Name', 'Team 1 Score', 'Team 2 Name', 'Team 2 Score', 'Score'])
y1 = (new_x['Score'] > 0).astype(int)

In [32]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC


def new_predict(X_train, y_train):
    from sklearn.ensemble import VotingClassifier
    from sklearn.linear_model import LogisticRegression
    
    # Create base models
    xgb_model = xgb.XGBClassifier(random_state=42, n_estimators=500)
    lr_model  = LogisticRegression(random_state=42)
    rf_model  = RandomForestClassifier(n_estimators=500, random_state=42)
    svm_model = SVC(probability=True, random_state=42)
    gbm_model = GradientBoostingClassifier(n_estimators=500, random_state=42)
    knn_model = KNeighborsClassifier(n_neighbors=5)
    
    # Create ensemble
    ensemble = VotingClassifier(
        estimators=[
            ('xgb', xgb_model),
            ('lr', lr_model),
            ('rf', rf_model),
            ('svm', svm_model),
            #('gbm', gbm_model),
            ('knn', knn_model),
            
        ],
        voting='soft'
    )
    
    # Train and evaluate
    fit = ensemble.fit(X_train, y_train)
    
    return fit

In [33]:
fit_2025 = new_predict(X1, y1)

In [34]:
acc, fit = ensemble_prediction(X1, y1, X2, y2)
print(acc)

0.9402985074626866


In [35]:
teams

,Team 1 Name,Team 2 Name
4,UConn,Stetson
5,Florida Atlantic,Northwestern
6,San Diego St.,UAB
7,Yale,Auburn
8,Duquesne,BYU
9,Illinois,Morehead St.
10,Washington St.,Drake
11,Iowa St.,South Dakota St.
12,Wagner,North Carolina
13,Mississippi St.,Michigan St.


In [36]:
matchups_2025

,Team 1 Name,Team 2 Name
0,Auburn,Alabama St.
1,Louisville,Creighton
2,Michigan,UC San Diego
3,Texas A&M,Yale
4,Ole Miss,North Carolina
5,Iowa St.,Lipscomb
6,Marquette,New Mexico
7,Michigan St.,Bryant
8,Florida,Norfolk St.
9,UConn,Oklahoma


In [37]:
ken_2025 = ken_2025.rename(columns={'Opp Non Conf NetRtg': 'Non Conf Opp NetRtg'})

In [38]:
run_simulation(matchups_2025, ken_2025, fit_2025)

Simulating Round 1...
Simulating Round 2...
Simulating Sweet 16...
Simulating Elite 8...
Simulating Final 4...
Simulating Championship...

🏀 MARCH MADNESS TOURNAMENT SIMULATION RESULTS 🏀


== Round 1 ==
Game 1: Auburn vs Alabama St.
Game 2: Louisville vs Creighton
Game 3: Michigan vs UC San Diego
Game 4: Texas A&M vs Yale
Game 5: Ole Miss vs North Carolina
Game 6: Iowa St. vs Lipscomb
Game 7: Marquette vs New Mexico
Game 8: Michigan St. vs Bryant
Game 9: Florida vs Norfolk St.
Game 10: UConn vs Oklahoma
Game 11: Memphis vs Colorado St.
Game 12: Maryland vs Grand Canyon
Game 13: Missouri vs Drake
Game 14: Texas Tech vs UNC Wilmington
Game 15: Kansas vs Arkansas
Game 16: St. John's vs Omaha
Game 17: Duke vs Mount St. Mary's
Game 18: Mississippi St. vs Baylor
Game 19: Oregon vs Liberty
Game 20: Arizona vs Akron
Game 21: BYU vs VCU
Game 22: Wisconsin vs Montana
Game 23: Saint Mary's vs Vanderbilt
Game 24: Alabama vs Robert Morris
Game 25: Houston vs SIUE
Game 26: Gonzaga vs Georgia
Game 27

{'Round 1':         Team 1 Name       Team 2 Name
 0            Auburn       Alabama St.
 1        Louisville         Creighton
 2          Michigan      UC San Diego
 3         Texas A&M              Yale
 4          Ole Miss    North Carolina
 5          Iowa St.          Lipscomb
 6         Marquette        New Mexico
 7      Michigan St.            Bryant
 8           Florida       Norfolk St.
 9             UConn          Oklahoma
 10          Memphis      Colorado St.
 11         Maryland      Grand Canyon
 12         Missouri             Drake
 13       Texas Tech    UNC Wilmington
 14           Kansas          Arkansas
 15       St. John's             Omaha
 16             Duke  Mount St. Mary's
 17  Mississippi St.            Baylor
 18           Oregon           Liberty
 19          Arizona             Akron
 20              BYU               VCU
 21        Wisconsin           Montana
 22     Saint Mary's        Vanderbilt
 23          Alabama     Robert Morris
 24          H